# 05 - Final Load Preparation
## Retail Analytics | SectionB Group 3

This notebook prepares the cleaned dataset for machine learning:
- Drop irrelevant identifier columns
- Label-encode all categorical variables
- Standard-scale numerical features
- Validate zero nulls
- Perform 80/20 train/test split
- Save final CSVs to `data/processed/`


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ── Resolve project root (works locally and on Colab)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH             = PROCESSED_DIR / 'cleaned_dataset.csv'
TABLEAU_READY_PATH     = PROCESSED_DIR / 'tableau_ready_dataset.csv'

df = pd.read_csv(INPUT_PATH)
print('Loaded:', df.shape)
df.head()


## 0. KPI Calculations & Tableau Export

All KPI calculations use `df_kpi` — the original cleaned dataset in human-readable form (not encoded). This keeps the ML pipeline (`df`) cleanly separated from the analytics layer.


### 0a. Overall KPIs


In [ ]:
# Load original cleaned data (before encoding) for KPI calculations
df_kpi = pd.read_csv(INPUT_PATH)
df_kpi['Date'] = pd.to_datetime(df_kpi['Date'])

total_revenue       = df_kpi['Total_Amount'].sum()
total_transactions  = df_kpi['Transaction_ID'].nunique()
total_customers     = df_kpi['Customer_ID'].nunique()
overall_aov         = df_kpi['Total_Amount'].mean()
avg_rating          = df_kpi['Ratings'].mean()

print('=' * 45)
print(f'  Total Revenue:            ₹{total_revenue:,.2f}')
print(f'  Total Transactions:        {total_transactions:,}')
print(f'  Unique Customers:          {total_customers:,}')
print(f'  Overall AOV:              ₹{overall_aov:,.2f}')
print(f'  Average Rating:            {avg_rating:.2f} / 5')
print('=' * 45)


### 0b. Revenue by Customer Segment


In [ ]:
revenue_by_segment = df_kpi.groupby('Customer_Segment')['Total_Amount'].sum().sort_values(ascending=False)
print('Total Revenue per Customer Segment (₹):\n', revenue_by_segment)

plt.figure(figsize=(8, 4))
revenue_by_segment.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Total Revenue per Customer Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Total Revenue (₹)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_revenue_by_segment.png')
plt.show()


### 0c. AOV by Product Category


In [ ]:
aov_by_category = df_kpi.groupby('Product_Category')['Total_Amount'].mean().sort_values(ascending=False)
print('Average Order Value by Product Category (₹):\n', aov_by_category)

plt.figure(figsize=(10, 4))
aov_by_category.plot(kind='bar', color='darkorange', edgecolor='black')
plt.title('Average Order Value (AOV) by Product Category (₹)')
plt.xlabel('Product Category')
plt.ylabel('AOV (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_aov_by_category.png')
plt.show()


### 0d. Sales Contribution % by Product Category


In [ ]:
sales_pct = (df_kpi.groupby('Product_Category')['Total_Amount'].sum() / total_revenue * 100).sort_values(ascending=False)
print('Sales Contribution (%) by Product Category:\n', sales_pct.round(2))

plt.figure(figsize=(8, 8))
plt.pie(sales_pct.values, labels=sales_pct.index, autopct='%1.1f%%', startangle=90)
plt.title('Sales Contribution (%) by Product Category')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_sales_pct_category.png')
plt.show()


### 0e. Monthly Revenue Trend


In [ ]:
monthly_revenue = df_kpi.groupby('Month')['Total_Amount'].sum()
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly_revenue = monthly_revenue.reindex([m for m in month_order if m in monthly_revenue.index])

plt.figure(figsize=(12, 4))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', color='teal', linewidth=2)
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_monthly_trend.png')
plt.show()


### 0f. Revenue by Country (Top 10)


In [ ]:
country_revenue = df_kpi.groupby('Country')['Total_Amount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 4))
country_revenue.plot(kind='barh', color='purple', edgecolor='black')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Total Revenue (₹)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_revenue_by_country.png')
plt.show()


### 0g. Order Status Breakdown


In [ ]:
order_status_pct = df_kpi['Order_Status'].value_counts(normalize=True) * 100
print('Order Status Distribution (%):\n', order_status_pct.round(2))

plt.figure(figsize=(7, 4))
order_status_pct.plot(kind='bar', color='crimson', edgecolor='black')
plt.title('Order Status Distribution (%)')
plt.xlabel('Status')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_order_status.png')
plt.show()


### 0h. Average Rating by Product Category


In [ ]:
avg_rating_cat = df_kpi.groupby('Product_Category')['Ratings'].mean().sort_values(ascending=False)
print('Average Rating by Product Category:\n', avg_rating_cat.round(2))

plt.figure(figsize=(10, 4))
avg_rating_cat.plot(kind='bar', color='gold', edgecolor='black')
plt.title('Average Rating by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Average Rating')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_avg_rating_category.png')
plt.show()


### 0i. Revenue by Payment Method


In [ ]:
payment_revenue = df_kpi.groupby('Payment_Method')['Total_Amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
payment_revenue.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Revenue by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Total Revenue (₹)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_revenue_by_payment.png')
plt.show()


### 0j. Weekend vs Weekday Revenue


In [ ]:
weekend_revenue = df_kpi.groupby('is_weekend')['Total_Amount'].sum()
weekend_revenue.index = ['Weekday', 'Weekend']

plt.figure(figsize=(6, 4))
weekend_revenue.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black')
plt.title('Weekday vs Weekend Revenue')
plt.ylabel('Total Revenue (₹)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_weekend_vs_weekday.png')
plt.show()


### 0k. Save Tableau-Ready Dataset


In [ ]:
# Save original cleaned data (human-readable, not encoded) for Tableau
df_kpi.to_csv(TABLEAU_READY_PATH, index=False)
print(f'✓ Tableau-ready dataset saved → {TABLEAU_READY_PATH}')
print(f'  Shape: {df_kpi.shape}')


**KPI Insights:** These KPIs form the foundation of the retail dashboard. Revenue by segment identifies which customer groups drive the most value. AOV by category highlights premium product lines. Monthly trend reveals seasonal patterns. Country revenue guides geographic marketing spend. Order status and ratings flag operational and quality issues that, if resolved, directly recover lost revenue.


## 1. Drop Irrelevant Columns


In [ ]:
# Drop identifier columns — not useful for modelling
drop_cols = ['Transaction_ID', 'Customer_ID']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print('Shape after dropping identifiers:', df.shape)


## 2. Label Encode Categorical Variables


In [ ]:
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()

# Convert bool columns first
bool_cols = df.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)

# Label encode all remaining object columns
obj_cols = df.select_dtypes(include='object').columns.tolist()
label_mappings = {}
for col in obj_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('Encoded columns:', obj_cols)
print('Shape:', df.shape)
print(df.dtypes)


## 3. Final Null Check


In [ ]:
null_count = df.isnull().sum().sum()
print(f'Total null values: {null_count}')
assert null_count == 0, f'Dataset still contains {null_count} null values — fix upstream cleaning step!'
print('✓ No null values found. Dataset is ready for modelling.')


## 4. Scale Numerical Features


In [ ]:
# Identify numerical columns to scale (exclude already-encoded categoricals and binary flags)
skip_scaling = ['order_success', 'is_weekend', 'day']
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in num_cols if c not in skip_scaling]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('Scaled columns:', scale_cols)
print(df[scale_cols].describe().round(3))


## 5. Train / Test Split (80 / 20)


In [ ]:
TARGET = 'order_success'
feature_drop = ['order_success', 'Date', 'Time', 'Month', 'day']
feature_cols = [c for c in df.columns if c not in feature_drop]

X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Target variable:  {TARGET}')
print(f'Features used:    {len(feature_cols)}')
print(f'Training set:     {X_train.shape[0]:,} rows')
print(f'Test set:         {X_test.shape[0]:,} rows')


## 6. Save Final Datasets


In [ ]:
# Save full prepared dataset
final_path = PROCESSED_DIR / 'final_prepared_dataset.csv'
df.to_csv(final_path, index=False)
print(f'Saved full prepared dataset → {final_path}')

# Save train set
train_df = X_train.copy()
train_df[TARGET] = y_train.values
train_path = PROCESSED_DIR / 'train_set.csv'
train_df.to_csv(train_path, index=False)
print(f'Saved training set          → {train_path}')

# Save test set
test_df = X_test.copy()
test_df[TARGET] = y_test.values
test_path = PROCESSED_DIR / 'test_set.csv'
test_df.to_csv(test_path, index=False)
print(f'Saved test set              → {test_path}')

print('\n✓ All files saved successfully.')


## 7. Final Summary


| Output File | Description |
|---|---|
| `cleaned_dataset.csv` | Cleaned input data |
| `tableau_ready_dataset.csv` | Human-readable export for Tableau dashboard |
| `final_prepared_dataset.csv` | Fully encoded and scaled ML-ready dataset |
| `train_set.csv` | 80% training split |
| `test_set.csv` | 20% test split |

**Target Variable:** `order_success` (1 = Completed order, 0 = otherwise)  
**Total KPIs Computed:** 10  
**Total Features for ML:** All encoded columns except target and date/time fields  
**Pipeline:** Extract → Clean → EDA → Statistical Analysis → Final Prep  


In [ ]:
print('Pipeline complete.')
print(f'  Processed dir: {PROCESSED_DIR}')
print(f'  Figures dir:   {FIGURES_DIR}')
print(f'  Outputs:')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'    {f.name:<35} {size_kb:>8.1f} KB')
